# Learning 03: Fine-Tuning GLiClass

Zero-shot GLiClass works well out of the box, but fine-tuning on domain-specific labeled data
can significantly improve accuracy. This notebook covers the full fine-tuning workflow:

1. **Data format** — `{text, all_labels, true_labels}`
2. **Dataset creation** — `GLiClassDataset` + `DataCollatorWithPadding`
3. **Training** — `TrainingArguments` + `Trainer`
4. **Evaluation** — before/after accuracy comparison

## Why fine-tune?

| Scenario | Zero-shot | Fine-tuned |
|----------|-----------|------------|
| No labeled data | ✓ | ✗ |
| < 100 examples | ✓ (faster) | Marginal gain |
| 100–1000 examples | ✓ OK | ✓ Better |
| 1000+ examples | OK | ✓✓ Significantly better |
| Domain-specific jargon | May miss | ✓ Learns it |

In [1]:
import json, os, random, warnings
warnings.filterwarnings('ignore')
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from gliclass.data_processing import GLiClassDataset, DataCollatorWithPadding, AugmentationConfig
from gliclass.training import TrainingArguments, Trainer
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "classification_training_data.json")) as f:
    data = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Loaded {len(data)} labeled examples, device={device}")

✓ Loaded 50 labeled examples, device=cuda


## 1. Training Data Format

GLiClass training data must follow this structure:

```json
{
  "text": "Apple unveiled the new M4 MacBook Pro...",
  "all_labels": ["technology", "finance", "sports", "science", "politics"],
  "true_labels": ["technology"]
}
```

- `all_labels`: **all candidate labels** the model should choose from
- `true_labels`: **the correct labels** for this example (can be multiple for multi-label)

In [2]:
# Inspect data format
print("Sample example:")
print(json.dumps(data[0], indent=2))

# Check class distribution
from collections import Counter
label_counts = Counter(d['true_labels'][0] for d in data)
print(f"\nClass distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:12}: {count} examples")

Sample example:
{
  "text": "Apple unveiled a new MacBook Pro powered by the M4 chip, delivering significant performance improvements over the previous generation. The laptop features a Liquid Retina XDR display and up to 24 hours of battery life.",
  "all_labels": [
    "technology",
    "finance",
    "sports",
    "science",
    "politics"
  ],
  "true_labels": [
    "technology"
  ]
}

Class distribution:
  finance     : 10 examples
  politics    : 10 examples
  science     : 10 examples
  sports      : 10 examples
  technology  : 10 examples


## 2. Load Model and Measure Zero-Shot Accuracy

In [3]:
model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)

# Shuffle and split: 80% train, 20% test
random.seed(42)
random.shuffle(data)
split = int(len(data) * 0.8)
train_data, eval_data = data[:split], data[split:]
print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

# Baseline: zero-shot accuracy before fine-tuning
LABELS = ["technology", "finance", "sports", "science", "politics"]
pipeline = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='single-label', device=device
)
eval_texts = [d['text'] for d in eval_data]
eval_true = [d['true_labels'][0] for d in eval_data]

baseline_results = pipeline(eval_texts, LABELS, batch_size=4)
baseline_preds = [r[0]['label'] for r in baseline_results]
acc_before = sum(p == t for p, t in zip(baseline_preds, eval_true)) / len(eval_true)
print(f"\nZero-shot accuracy: {acc_before:.0%} ({sum(p==t for p,t in zip(baseline_preds,eval_true))}/{len(eval_true)})")

Train: 40 | Eval: 10


  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:02<00:05,  2.55s/it]

100%|██████████| 3/3 [00:02<00:00,  1.17it/s]


Zero-shot accuracy: 80% (8/10)


## 3. Create Datasets

`GLiClassDataset` tokenizes and prepares examples for training.

Key parameters:
- `problem_type='multi_label_classification'` — use even for single-label (required for `DataCollatorWithPadding` compatibility)
- `architecture_type='uni-encoder'` — matches the edge model architecture
- `prompt_first=True` — prepend label tokens before text
- `AugmentationConfig(enabled=False)` — disable augmentation for eval

In [4]:
augment_config = AugmentationConfig(enabled=False)

train_dataset = GLiClassDataset(
    train_data, tokenizer, augment_config,
    max_length=256,
    problem_type='multi_label_classification',
    architecture_type='uni-encoder',
    prompt_first=True
)
eval_dataset = GLiClassDataset(
    eval_data, tokenizer, augment_config,
    max_length=256,
    problem_type='multi_label_classification',
    architecture_type='uni-encoder',
    prompt_first=True
)
data_collator = DataCollatorWithPadding(device=device)

sample = train_dataset[0]
print(f"Dataset item keys: {list(sample.keys())}")
print(f"  labels shape: {sample['labels'].shape}  (one-hot over {len(LABELS)} classes)")
print(f"  labels_text: {sample['labels_text']}")
print(f"  Train batches: {len(train_dataset)} | Eval batches: {len(eval_dataset)}")

Total labels:  5
Total labels:  5
Dataset item keys: ['input_ids', 'attention_mask', 'labels', 'labels_text', 'input_texts']
  labels shape: torch.Size([5])  (one-hot over 5 classes)
  labels_text: ['sports', 'science', 'finance', 'technology', 'politics']
  Train batches: 40 | Eval batches: 10


## 4. Fine-Tune with Trainer

Key `TrainingArguments`:

| Parameter | Meaning |
|-----------|--------|
| `learning_rate` | Encoder LR (small: 1e-5 to 5e-5) |
| `others_lr` | Classifier head LR (larger: 1e-4) |
| `max_steps` | Total training steps |
| `per_device_train_batch_size` | Batch size per GPU/CPU |
| `eval_strategy` | When to evaluate ("steps" or "epoch") |

In [5]:
import tempfile

output_dir = tempfile.mkdtemp(prefix="gliclass_ft_")

def compute_metrics(p):
    preds, labels = p
    preds_flat = (preds.reshape(-1) > 0.5).astype(int)
    labels_flat = (labels.reshape(-1) > 0.5).astype(int)
    return {"accuracy": float(accuracy_score(labels_flat, preds_flat))}

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    max_steps=150,
    learning_rate=3e-5,
    others_lr=1e-4,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=75,
    save_steps=1000,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Fine-tuning for {training_args.max_steps} steps...")
trainer.train()
print("✓ Training complete")

Fine-tuning for 150 steps...


Step,Training Loss,Validation Loss,Accuracy
75,0.153000,0.092188,0.940000
150,0.000000,0.003353,0.980000


✓ Training complete


## 5. Evaluate and Compare

In [6]:
# Accuracy after fine-tuning using the same pipeline
# (model weights are updated in-place by trainer)
ft_results = pipeline(eval_texts, LABELS, batch_size=4)
ft_preds = [r[0]['label'] for r in ft_results]
acc_after = sum(p == t for p, t in zip(ft_preds, eval_true)) / len(eval_true)

print(f"Zero-shot accuracy:  {acc_before:.0%}")
print(f"Fine-tuned accuracy: {acc_after:.0%}")
print(f"Improvement:         +{acc_after - acc_before:.0%}")
print()
print("Predictions:")
for pred, true, text in zip(ft_preds, eval_true, eval_texts):
    mark = "✓" if pred == true else "✗"
    print(f"  {mark} [{pred:12}] {text[:60]}...")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 93.29it/s]

Zero-shot accuracy:  80%
Fine-tuned accuracy: 100%
Improvement:         +20%

Predictions:
  ✓ [technology  ] A new open-source database engine written in Rust achieves 1...
  ✓ [science     ] A study published in Nature demonstrated that octopuses expe...
  ✓ [technology  ] Tesla released a major over-the-air update adding full self-...
  ✓ [technology  ] Quantum computing company IonQ demonstrated a 40-qubit proce...
  ✓ [finance     ] Crude oil prices dropped to a six-month low amid rising US i...
  ✓ [finance     ] Hedge fund Bridgewater Associates reported its All Weather p...
  ✓ [finance     ] Venture capital funding for AI startups reached $25 billion ...
  ✓ [technology  ] Google DeepMind released a new large language model that out...
  ✓ [technology  ] A new programming language designed for AI systems promises ...
  ✓ [politics    ] The US Senate passed a bipartisan infrastructure bill alloca...


## 6. Save the Fine-Tuned Model

In [7]:
import os
final_dir = os.path.join(output_dir, "final_model")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✓ Model saved to {final_dir}")

# Reload and verify
model_reloaded = GLiClassModel.from_pretrained(final_dir)
pipeline_reloaded = ZeroShotClassificationPipeline(
    model_reloaded, tokenizer, classification_type='single-label', device=device
)
test_result = pipeline_reloaded("Apple launched the new iPhone 17 with a satellite connectivity feature.", LABELS)[0]
print(f"\nReloaded model prediction: '{test_result[0]['label']}' ({test_result[0]['score']:.2f})")

print("\n# Summary")
print("# Fine-tuning workflow:")
print("# 1. Prepare data: {text, all_labels, true_labels}")
print("# 2. Create GLiClassDataset (problem_type='multi_label_classification')")
print("# 3. Configure TrainingArguments (learning_rate + others_lr)")
print("# 4. Train with Trainer")
print("# 5. Save with model.save_pretrained()")

✓ Model saved to /tmp/gliclass_ft_xlu8r3vc/final_model

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.37it/s]

100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


Reloaded model prediction: 'technology' (1.00)

# Summary
# Fine-tuning workflow:
# 1. Prepare data: {text, all_labels, true_labels}
# 2. Create GLiClassDataset (problem_type='multi_label_classification')
# 3. Configure TrainingArguments (learning_rate + others_lr)
# 4. Train with Trainer
# 5. Save with model.save_pretrained()
